In [1]:
import pandas as pd
import os
from PIL import Image
import shutil

In [14]:
# List of NORMAL and PNEUMONIA folder paths
normal_folders = [
    r'../raw/Chest X-Ray Images (Pneumonia)/test/NORMAL',
    r'../raw/Chest X-Ray Images (Pneumonia)/train/NORMAL',
    r'../raw/Chest X-Ray Images (Pneumonia)/val/NORMAL'
]

pneumonia_folders = [
    r'../raw/Chest X-Ray Images (Pneumonia)/test/PNEUMONIA',
    r'../raw/Chest X-Ray Images (Pneumonia)/train/PNEUMONIA',
    r'../raw/Chest X-Ray Images (Pneumonia)/val/PNEUMONIA'
]

# Get all filenames in NORMAL folders
normal_files = []
for folder in normal_folders:
    normal_files.extend([os.path.join(folder, f) for f in os.listdir(folder) if os.path.isfile(os.path.join(folder, f))])

# Get all filenames in PNEUMONIA folders
pneumonia_files = []
for folder in pneumonia_folders:
    pneumonia_files.extend([os.path.join(folder, f) for f in os.listdir(folder) if os.path.isfile(os.path.join(folder, f))])

# Read file name and dimensions for each image in normal_files
normal_file_info = []
for file_path in normal_files:
    if not file_path.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.gif')):
        continue
    try:
        with Image.open(file_path) as img:
            width, height = img.size
        normal_file_info.append({'file_name': os.path.basename(file_path), 'width': width, 'height': height, 'classification': 'Normal'})
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

# Read file name and dimensions for each image in pneumonia_files
pneumonia_file_info = []
for file_path in pneumonia_files:
    if not file_path.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.gif')):
        continue
    try:
        with Image.open(file_path) as img:
            width, height = img.size
        pneumonia_file_info.append({'file_name': os.path.basename(file_path), 'width': width, 'height': height, 'classification': 'Pneumonia'})
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

# Combine both lists into a DataFrame
all_file_info = normal_file_info + pneumonia_file_info
df = pd.DataFrame(all_file_info)

# Display the first 5 rows as a sample
df.head()

,file_name,width,height,classification
0,IM-0001-0001.jpeg,1857,1317,Normal
1,IM-0003-0001.jpeg,2111,1509,Normal
2,IM-0005-0001.jpeg,2031,1837,Normal
3,IM-0006-0001.jpeg,1663,1326,Normal
4,IM-0007-0001.jpeg,2053,1818,Normal


In [17]:
df.to_csv('../processed/Pneumonia/metadata.csv', index=False)

In [20]:
# Destination folder
dest_folder = '../processed/Pneumonia/Images'
os.makedirs(dest_folder, exist_ok=True)

# List of all source folders
all_folders = [
    r'../raw/Chest X-Ray Images (Pneumonia)/test/NORMAL',
    r'../raw/Chest X-Ray Images (Pneumonia)/train/NORMAL',
    r'../raw/Chest X-Ray Images (Pneumonia)/val/NORMAL',
    r'../raw/Chest X-Ray Images (Pneumonia)/test/PNEUMONIA',
    r'../raw/Chest X-Ray Images (Pneumonia)/train/PNEUMONIA',
    r'../raw/Chest X-Ray Images (Pneumonia)/val/PNEUMONIA'
]

# Copy files
for folder in all_folders:
    for file_name in os.listdir(folder):
        src_path = os.path.join(folder, file_name)
        if os.path.isfile(src_path):
            # Only copy image files
            if file_name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.gif')):
                dest_path = os.path.join(dest_folder, file_name)
                shutil.copy2(src_path, dest_path)

print("All files copied to", dest_folder)

All files copied to ../processed/Pneumonia/Images


In [29]:
md = pd.read_csv('../processed/Pneumonia/metadata.csv')
md_1 = pd.read_csv('../processed/Pneumonia/metadata_1.csv')

In [30]:
md.head()

,file_name,width,height,classification
0,IM-0001-0001.jpeg,1857,1317,Normal
1,IM-0003-0001.jpeg,2111,1509,Normal
2,IM-0005-0001.jpeg,2031,1837,Normal
3,IM-0006-0001.jpeg,1663,1326,Normal
4,IM-0007-0001.jpeg,2053,1818,Normal


In [31]:
md_1.head()

,FILE NAME,FORMAT,SIZE,URL,classification
0,Viral Pneumonia-1,PNG,256*256,https://www.kaggle.com/paultimothymooney/chest...,Pneumonia
1,Viral Pneumonia-2,PNG,256*256,https://www.kaggle.com/paultimothymooney/chest...,Pneumonia
2,Viral Pneumonia-3,PNG,256*256,https://www.kaggle.com/paultimothymooney/chest...,Pneumonia
3,Viral Pneumonia-4,PNG,256*256,https://www.kaggle.com/paultimothymooney/chest...,Pneumonia
4,Viral Pneumonia-5,PNG,256*256,https://www.kaggle.com/paultimothymooney/chest...,Pneumonia


In [32]:
# Assuming md_1 is a DataFrame with columns 'FILE NAME' and 'FORMAT'
# Merge them into a new column 'file_name' (e.g., 'Viral Pneumonia-1.png')
if 'FILE NAME' in md_1.columns and 'FORMAT' in md_1.columns:
    md_1['file_name'] = md_1['FILE NAME'].astype(str) + '.' + md_1['FORMAT'].astype(str).str.lower()
    print(md_1[['FILE NAME', 'FORMAT', 'file_name']].head())
else:
    print("md_1 does not have the required columns 'FILE NAME' and 'FORMAT'.")

           FILE NAME FORMAT              file_name
0  Viral Pneumonia-1    PNG  Viral Pneumonia-1.png
1  Viral Pneumonia-2    PNG  Viral Pneumonia-2.png
2  Viral Pneumonia-3    PNG  Viral Pneumonia-3.png
3  Viral Pneumonia-4    PNG  Viral Pneumonia-4.png
4  Viral Pneumonia-5    PNG  Viral Pneumonia-5.png


In [38]:
# Split 'Size' column in md_1 into 'width' and 'height' columns
if 'SIZE' in md_1.columns:
    md_1[['width', 'height']] = md_1['SIZE'].str.split('*', expand=True).astype(int)
    print(md_1[['SIZE', 'width', 'height']].head())
else:
    print("md_1 does not have a 'SIZE' column.")

      SIZE  width  height
0  256*256    256     256
1  256*256    256     256
2  256*256    256     256
3  256*256    256     256
4  256*256    256     256


In [40]:
md_1 = md_1[['file_name', 'width', 'height', 'classification']]

In [41]:
pneumonia_metadata = pd.concat([md, md_1], ignore_index=True)

In [44]:
import os

# Delete old metadata files if they exist
for old_file in [
    '../processed/Pneumonia/metadata_1.csv',
    '../processed/Pneumonia/metadata.csv'
]:
    if os.path.exists(old_file):
        os.remove(old_file)
        print(f"Deleted {old_file}")
    else:
        print(f"File not found: {old_file}")

# Save pneumonia_metadata as metadata.csv
pneumonia_metadata.to_csv('../processed/Pneumonia/metadata.csv', index=False)
print("Saved pneumonia_metadata as metadata.csv")

Deleted ../processed/Pneumonia/metadata_1.csv
Deleted ../processed/Pneumonia/metadata.csv
Saved pneumonia_metadata as metadata.csv


In [16]:
df = pd.read_csv("../processed/all_data/metadata.csv")

In [19]:
# Filter Pneumonia rows
pneumonia_mask = df["classification"] == "Pneumonia"
pneumonia_df = df.loc[pneumonia_mask].copy()

# Assign incremental patient IDs: PN1, PN2, PN3, ...
pneumonia_df["patient_id"] = [f"PN{i}" for i in range(1, len(pneumonia_df) + 1)]

# Write back into the main dataframe
df.loc[pneumonia_mask, "patient_id"] = pneumonia_df["patient_id"].values

# Preview Pneumonia rows with new patient IDs
df.loc[pneumonia_mask, ["file_name", "classification", "patient_id"]].head(20)

,file_name,classification,patient_id
15391,person100_bacteria_475.jpeg,Pneumonia,PN1
15392,person100_bacteria_477.jpeg,Pneumonia,PN2
15393,person100_bacteria_478.jpeg,Pneumonia,PN3
15394,person100_bacteria_479.jpeg,Pneumonia,PN4
15395,person100_bacteria_480.jpeg,Pneumonia,PN5
15396,person100_bacteria_481.jpeg,Pneumonia,PN6
15397,person100_bacteria_482.jpeg,Pneumonia,PN7
15398,person101_bacteria_483.jpeg,Pneumonia,PN8
15399,person101_bacteria_484.jpeg,Pneumonia,PN9
15400,person101_bacteria_485.jpeg,Pneumonia,PN10


In [18]:
# Save updated metadata with new Pneumonia patient IDs
df.to_csv("../processed/all_data/metadata.csv", index=False)
print("Updated metadata saved to ../processed/all_data/metadata.csv")

Updated metadata saved to ../processed/all_data/metadata.csv
